# 🎮 Steam Video Game Recommender — Hybrid (Content-Based + Collaborative Filtering)

**Goal:** recommend Steam games a user is likely to enjoy, by blending two signals:

| Component | Signal | Method |
|---|---|---|
| **Content-based (CB)** | What a game *is* — community tags + store description | TF-IDF vectors + cosine similarity |
| **Collaborative filtering (CF)** | What similar players *actually liked* — positive reviews weighted by hours played | Item-item kNN |
| **Hybrid** | Both | `score = α·CF + (1−α)·CB` (per-user min-max normalised) |

**Dataset:** [Game Recommendations on Steam](https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam) (Kaggle, Anton Kozyriev)
- `recommendations.csv` — ~41M user reviews: user, game, **recommended yes/no**, **hours played**
- `games.csv` — ~50k games: title, release date, rating, price, platforms
- `games_metadata.json` — description + **user tags** for every game

**Pipeline:** Load → Clean & filter → EDA → Train/test split → CB model → CF model → Hybrid → Evaluate (Precision/Recall/NDCG/Hit-rate@K vs popularity baseline) → Tune α → Demo → Export model for the Streamlit app.

> Works in Google Colab and local Jupyter. First run downloads ~700 MB (zipped) from Kaggle. Games released up to **October 2023**.

## 1. Setup

In [ ]:
import os, glob, time, shutil
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import joblib

pd.set_option("display.max_colwidth", 80)
RNG = np.random.default_rng(42)
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
print("Ready")

## 2. Get the data
Downloads the dataset with `kagglehub`. Public datasets usually download without login; if it asks for credentials, either
- run `kagglehub.login()` (paste your username + API key from kaggle.com → Settings → *Create New Token*), **or**
- download the zip manually from the Kaggle page, unzip it, and put the 4 files in a folder called `data/` next to this notebook (in Colab: upload them via the 📁 Files panel). The cell skips the download if the files are already there.

In [ ]:
NEEDED = ["recommendations.csv", "games.csv", "games_metadata.json"]

def have_files():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in NEEDED)

if not have_files():
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    path = kagglehub.dataset_download("antonkozyriev/game-recommendations-on-steam")
    for f in NEEDED + ["users.csv"]:
        src = os.path.join(path, f)
        if os.path.exists(src) and not os.path.exists(os.path.join(DATA_DIR, f)):
            os.symlink(os.path.abspath(src), os.path.join(DATA_DIR, f))

assert have_files(), f"Missing files in {DATA_DIR}/ — see the instructions above."
for f in NEEDED:
    print(f"{f:<25} {os.path.getsize(os.path.join(DATA_DIR, f))/1e6:8.1f} MB")

### Load
`recommendations.csv` is large (~41M rows, ~2 GB). We read only the 4 columns we need with compact data types (int32 / float32 / bool) so it fits comfortably in Colab's free RAM.

In [ ]:
t0 = time.time()
reviews = pd.read_csv(os.path.join(DATA_DIR, "recommendations.csv"),
                      usecols=["user_id", "app_id", "is_recommended", "hours"],
                      dtype={"user_id": "int32", "app_id": "int32", "hours": "float32"})
reviews["is_recommended"] = reviews["is_recommended"].astype(str).str.lower().isin(["true", "1"])
print(f"Reviews loaded in {time.time()-t0:.0f}s: {len(reviews):,} rows | "
      f"{reviews.user_id.nunique():,} users | {reviews.app_id.nunique():,} games")
print(f"Positive reviews: {reviews.is_recommended.mean():.1%}")
reviews.head()

In [ ]:
games = pd.read_csv(os.path.join(DATA_DIR, "games.csv"))
meta = pd.read_json(os.path.join(DATA_DIR, "games_metadata.json"), lines=True)
games = games.merge(meta, on="app_id", how="left")
games["tags"] = games["tags"].apply(lambda x: x if isinstance(x, list) else [])
games["description"] = games["description"].fillna("")
print(f"{len(games):,} games | {games.tags.str.len().gt(0).mean():.1%} have tags")
print("Release dates:", games.date_release.min(), "→", games.date_release.max())
games[["app_id", "title", "date_release", "rating", "positive_ratio", "price_final", "tags"]].head()

## 3. Clean & filter
Design decisions (each one is a talking point in the viva):
1. **Use only positive reviews (`is_recommended = True`).** A negative review means the player tried the game and *didn't* like it — treating it as a "like" would teach the model the wrong thing.
2. **Implicit + explicit feedback → one weight:** `w = 1 + log(1 + hours)`. The `1` says "this player recommended it"; `log(1+hours)` adds how much they played. The log stops a 5,000-hour player from dominating, and the `+1` keeps reviews with 0 hours recorded from vanishing.
3. **k-core filtering:** keep users with ≥ 5 liked games and games with ≥ 10 such users. Most of the 14M users wrote only 1–2 reviews — there's nothing to learn about their taste and nothing to evaluate.
4. **Drop non-games** — soundtracks, season passes, DLC add-ons and artbooks (matched by title). Otherwise the model happily recommends "*Fallout 4 Season Pass*" to Fallout 4 fans, which is technically correct but useless.
5. **Cap on users (`MAX_USERS`)** so the notebook runs on free Colab in ~20 minutes. Raise it if you have more RAM/time.

In [ ]:
MIN_USER_GAMES, MIN_GAME_USERS, MAX_USERS = 5, 10, 200_000

def kcore(df):
    # iterate until stable (removing games can push users below the threshold and vice versa)
    while True:
        n = len(df)
        g = df.app_id.value_counts();  df = df[df.app_id.isin(g[g >= MIN_GAME_USERS].index)]
        u = df.user_id.value_counts(); df = df[df.user_id.isin(u[u >= MIN_USER_GAMES].index)]
        if len(df) == n:
            return df

NON_GAME = r"(?i)\bsoundtrack\b|\bOST\b|season pass|artbook|art book|expansion pack|supporter pack|gold upgrade|premium upgrade| - .*\bDLC\b|\bDLC$"
non_game_ids = set(games.loc[games.title.str.contains(NON_GAME, na=False), "app_id"])
print(f"Non-game items (soundtracks, DLC, season passes): {len(non_game_ids):,}")

inter = reviews[reviews.is_recommended & ~reviews.app_id.isin(non_game_ids)]
inter = inter.drop_duplicates(["user_id", "app_id"])[["user_id", "app_id", "hours"]]
print(f"Positive reviews: {len(inter):,}")
inter = kcore(inter)
print(f"After k-core: {len(inter):,} rows | {inter.user_id.nunique():,} users | {inter.app_id.nunique():,} games")

users_all = inter.user_id.unique()
if len(users_all) > MAX_USERS:
    keep = RNG.choice(users_all, MAX_USERS, replace=False)
    inter = kcore(inter[inter.user_id.isin(keep)])
    print(f"Sampled {MAX_USERS:,} users → {len(inter):,} rows | {inter.user_id.nunique():,} users | {inter.app_id.nunique():,} games")

inter = inter.copy()
inter["weight"] = (1 + np.log1p(inter.hours.clip(lower=0))).astype("float32")
n_users, n_items = inter.user_id.nunique(), inter.app_id.nunique()
print(f"Matrix density: {len(inter)/(n_users*n_items):.4%}")
# keep the "not recommended" reviews: used later to learn what players dislike (section 10b)
neg_reviews = reviews.loc[~reviews.is_recommended, ["user_id", "app_id"]].copy()
print(f"Negative reviews kept for later: {len(neg_reviews):,}")
del reviews

### Game catalogue

In [ ]:
stats = inter.groupby("app_id").agg(players=("user_id", "size"), median_hours=("hours", "median")).reset_index()
catalog = stats.merge(games, on="app_id", how="left")
catalog["title"] = (catalog["title"].fillna("Unknown").str.replace(r"[™®©]", "", regex=True)
                    .str.replace(r"\s+", " ", regex=True).str.strip())   # "FC™ 25" -> "FC 25" so search works
catalog["tags"] = catalog["tags"].apply(lambda x: x if isinstance(x, list) else [])
catalog["description"] = catalog["description"].fillna("")
catalog["item_id"] = catalog["app_id"].astype(str)
catalog["genres"] = [[] for _ in range(len(catalog))]   # this dataset folds genres into tags
catalog["developer"] = np.nan
catalog["median_hours"] = catalog.median_hours.astype(float).round(1)
catalog = catalog.sort_values("app_id").reset_index(drop=True)
print(f"Catalogue: {len(catalog):,} games, {catalog.tags.str.len().gt(0).mean():.1%} have tags")
catalog.nlargest(10, "players")[["title", "date_release", "players", "median_hours", "rating"]]

### Fill in missing tags from SteamSpy
**Data-quality issue found during development:** in this dataset the metadata file has *empty* tags and descriptions for most of the biggest games — only ~15% of games with 100k+ reviews have tags (ELDEN RING, Cyberpunk 2077, GTA V and Dota 2 have none). Without a fix, the content-based model is blind exactly on the games users search for most.

**Fix:** for catalogue games with no tags, fetch their community tags and genres from the free [SteamSpy API](https://steamspy.com/api.php), most-played games first. SteamSpy allows ~1 request/second, so we cap it at `ENRICH_MAX` games and **cache** the results in `data/steamspy_tags.json` — re-runs are instant. If the network fails, the notebook simply carries on without it.

In [ ]:
import json, urllib.request

ENRICH_MAX = 800          # ~1 game per second; raise for better coverage, set 0 to skip
CACHE = os.path.join(DATA_DIR, "steamspy_tags.json")
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def steamspy(app_id):
    url = f"https://steamspy.com/api.php?request=appdetails&appid={app_id}"
    with urllib.request.urlopen(url, timeout=15) as r:
        d = json.loads(r.read().decode("utf-8"))
    tags = list(d.get("tags") or {})                     # already sorted by votes
    genres = [g.strip() for g in (d.get("genre") or "").split(",") if g.strip()]
    return {"tags": tags[:20], "genres": genres}

missing = catalog[catalog.tags.str.len() == 0].sort_values("players", ascending=False)
todo = [a for a in missing.item_id.head(ENRICH_MAX) if a not in cache]
print(f"Games without tags: {len(missing):,} | fetching {len(todo):,} from SteamSpy (cached: {len(cache):,})")

t0, fails = time.time(), 0
for n, app in enumerate(todo, 1):
    try:
        cache[app] = steamspy(app)
    except Exception:
        fails += 1
        if fails >= 10 and fails == n:               # network clearly unavailable -> stop early
            print("SteamSpy not reachable — continuing without enrichment.")
            break
    if n % 50 == 0:
        json.dump(cache, open(CACHE, "w"))
        print(f"  {n}/{len(todo)}  ({time.time()-t0:.0f}s)")
    time.sleep(1.0)
json.dump(cache, open(CACHE, "w"))

def merged_tags(row):
    if len(row.tags):
        return row.tags
    c = cache.get(row.item_id)
    return (c["tags"] + [g for g in c["genres"] if g not in c["tags"]]) if c else []

catalog["tags"] = catalog.apply(merged_tags, axis=1)
top100 = catalog.nlargest(100, "players")
print(f"Tag coverage now: {catalog.tags.str.len().gt(0).mean():.1%} of catalogue | "
      f"{top100.tags.str.len().gt(0).mean():.0%} of the 100 most-liked games")

In [ ]:
# Is a specific game in the final catalogue? (change the search text)
SEARCH = "FC 24"   # dataset ends Oct 2023, so FC 25 (Sept 2024) is not in it
catalog[catalog.title.str.contains(SEARCH, case=False, regex=False)][["title", "date_release", "players"]]

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.log10(inter.hours + 1), bins=60, color="#4c72b0")
ax[0].set(title="Hours played per liked game (log10)", xlabel="log10(hours + 1)")
ax[1].hist(inter.user_id.value_counts(), bins=np.logspace(np.log10(MIN_USER_GAMES), 3, 50), color="#55a868")
ax[1].set(xscale="log", title="Liked games per user", xlabel="# games")
ax[2].hist(inter.app_id.value_counts(), bins=np.logspace(np.log10(MIN_GAME_USERS), 5, 50), color="#c44e52")
ax[2].set(xscale="log", title="Players per game (long tail)", xlabel="# players")
plt.tight_layout(); plt.show()

top = catalog.nlargest(15, "players")[["title", "players"]][::-1]
tag_counts = Counter(t for ts in catalog.tags for t in ts).most_common(15)[::-1]
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].barh(top.title, top.players, color="#4c72b0"); ax[0].set_title("Most-liked games")
ax[1].barh([t for t, _ in tag_counts], [c for _, c in tag_counts], color="#8172b2"); ax[1].set_title("Most common tags")
plt.tight_layout(); plt.show()

years = pd.to_datetime(catalog.date_release, errors="coerce").dt.year.value_counts().sort_index()
years.plot.bar(figsize=(14, 3), color="#937860", title="Games in catalogue by release year"); plt.tight_layout(); plt.show()

**What the EDA tells us:** hours played are heavily skewed (hence the log), and a small head of games is liked by a huge share of users. That long tail is why a **popularity baseline is hard to beat** on accuracy — and why we also track **catalogue coverage**.

## 5. Index mapping & user-item matrix
Sparse matrix `R` (users × games), value = `1 + log(1 + hours)`.

In [ ]:
item_ids = catalog.item_id.tolist()
app_to_idx = {a: i for i, a in enumerate(catalog.app_id)}
user_ids = [str(u) for u in sorted(inter.user_id.unique())]
user_idx = {u: i for i, u in enumerate(user_ids)}
inter["u"] = inter.user_id.astype(str).map(user_idx).astype("int32")
inter["i"] = inter.app_id.map(app_to_idx).astype("int32")
title_of = dict(enumerate(catalog.title))
popularity = catalog.players.to_numpy().astype(float)
print("Users:", len(user_ids), "| Games:", len(item_ids))

## 6. Train / test split
Per user, hold out **20% of their liked games** (at least 1) as the test set. The model only sees the remaining 80% and must rank the hidden games high among *all games the user hasn't liked in training*.

In [ ]:
TEST_FRAC = 0.2
inter = inter.sample(frac=1.0, random_state=42)  # shuffle
inter["rank_in_user"] = inter.groupby("u").cumcount()
inter["n_user"] = inter.groupby("u").u.transform("size")
is_test = inter.rank_in_user < np.maximum(1, (inter.n_user * TEST_FRAC).round()).astype(int)
train, test = inter[~is_test], inter[is_test]

def to_matrix(df):
    return sp.csr_matrix((df.weight.to_numpy(), (df.u.to_numpy(), df.i.to_numpy())),
                         shape=(len(user_ids), len(item_ids)), dtype=np.float32)

R_train = to_matrix(train)
R_full = to_matrix(inter)
test_items = test.groupby("u").i.apply(set).to_dict()
print(f"Train rows: {len(train):,} | Test rows: {len(test):,} | Test users: {len(test_items):,}")

## 7. Content-based model (TF-IDF over tags + description)
Two TF-IDF views of every game, stacked side by side:
- **Tags** — community-voted labels (*Open World, Souls-like, Football…*). Multi-word tags are joined with `_` so *Open World* stays one token. This is the strongest content signal.
- **Description** — the store blurb, with English stop-words removed; weighted at 0.5 so it adds detail without overpowering the tags.

A user profile = weighted average of the vectors of games they liked. Score of a candidate game = cosine(profile, game).

In [ ]:
def tok(xs):
    return " ".join(str(x).strip().lower().replace(" ", "_").replace("-", "_") for x in xs if str(x).strip())

tfidf_tags = TfidfVectorizer(token_pattern=r"[^ ]+", min_df=2, sublinear_tf=True)
X_tags = tfidf_tags.fit_transform(catalog.tags.apply(tok))
tfidf_desc = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5, max_features=20000, sublinear_tf=True)
X_desc = tfidf_desc.fit_transform(catalog.description)
X_content = normalize(sp.hstack([X_tags, 0.5 * X_desc]).tocsr()).astype(np.float32)
print("Tag vocab:", X_tags.shape[1], "| Description vocab:", X_desc.shape[1], "| Content matrix:", X_content.shape)

def cb_scores(R_users):
    # R_users: (n × games) weighted history -> (n × games) cosine scores
    profiles = normalize(R_users @ X_content)
    return np.asarray((profiles @ X_content.T).todense())

def find_game(title):
    hits = catalog.index[catalog.title.str.lower() == title.lower()]
    return hits[0] if len(hits) else catalog.index[catalog.title.str.contains(title, case=False, regex=False)][0]

def similar_by_content(game_title, k=10):
    i = find_game(game_title)
    sims = (X_content @ X_content[i].T).toarray().ravel(); sims[i] = -1
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"game": catalog.title.values[top], "similarity": sims[top].round(3)})

similar_by_content(catalog.nlargest(1, "players").title.iloc[0])

## 8. Collaborative filtering (item-item kNN)
Two games are similar if the **same players like both**.
1. Item vectors = columns of `R_train` (who liked it, how much they played).
2. Cosine similarity between every pair of games, computed in chunks.
3. **Shrinkage:** multiply by `common / (common + 10)` so similarities backed by only a handful of shared players are toned down.
4. Keep only the **top-K = 50 neighbours** per game (sparse, denoised — and small enough for the web app).
5. User score for game *j* = Σ over liked games *i* of `w_ui · sim(i, j)`.

In [ ]:
def item_knn(R, k=50, chunk=500, shrink=10):
    # Top-k item-item cosine similarity with shrinkage for low-support pairs.
    Rn = normalize(R.T.tocsr())                  # games × users, L2 rows
    B = (R.T > 0).astype(np.float32).tocsr()     # binary: for co-occurrence counts
    n = Rn.shape[0]; rows, cols, vals = [], [], []
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        sim = (Rn[s:e] @ Rn.T).toarray()
        co = (B[s:e] @ B.T).toarray()
        sim *= co / (co + shrink)
        sim[np.arange(e - s), np.arange(s, e)] = 0
        kk = min(k, n - 1)
        top = np.argpartition(-sim, kk, axis=1)[:, :kk]
        for r in range(e - s):
            v = sim[r, top[r]]; keep = v > 0
            rows.extend([s + r] * keep.sum()); cols.extend(top[r][keep]); vals.extend(v[keep])
    return sp.csr_matrix((vals, (rows, cols)), shape=(n, n), dtype=np.float32)

t0 = time.time()
S_cf = item_knn(R_train, k=50)
print(f"Item-kNN built in {time.time()-t0:.0f}s | non-zeros: {S_cf.nnz:,}")

def cf_scores(R_users, S=None):
    S = S_cf if S is None else S
    return np.asarray((R_users @ S).todense())

def similar_by_cf(game_title, k=10, S=None):
    S = S_cf if S is None else S
    i = find_game(game_title)
    row = S[i].toarray().ravel()
    top = np.argsort(-row)[:k]
    return pd.DataFrame({"game": catalog.title.values[top], "cf_similarity": row[top].round(3)})

similar_by_cf(catalog.nlargest(1, "players").title.iloc[0])

## 9. Hybrid model
CB and CF scores live on different scales, so each is **min-max normalised per user** before blending:

$$\text{score}_{u,j} = \alpha \cdot \widetilde{CF}_{u,j} + (1-\alpha) \cdot \widetilde{CB}_{u,j}$$

α = 1 → pure CF, α = 0 → pure CB. We pick α on the evaluation below.

In [ ]:
def minmax_rows(M):
    lo = M.min(axis=1, keepdims=True); hi = M.max(axis=1, keepdims=True)
    return (M - lo) / np.where(hi - lo == 0, 1, hi - lo)

def hybrid_scores(R_users, alpha=0.7):
    return alpha * minmax_rows(cf_scores(R_users)) + (1 - alpha) * minmax_rows(cb_scores(R_users))

## 10. Evaluation
For each test user we rank all games not in their training history and check where the hidden games land.

| Metric | Meaning |
|---|---|
| **Precision@K** | fraction of the top-K that are hidden games |
| **Recall@K** | fraction of hidden games found in the top-K |
| **NDCG@K** | like recall but rewards ranking hits higher |
| **HitRate@K** | % users with ≥ 1 hit in the top-K |
| **Coverage** | % of the catalogue that appears in *anyone's* top-K (diversity) |

To keep it fast we evaluate on a random sample of up to 5,000 test users.

In [ ]:
K = 10
eval_users = np.array(sorted(test_items))
if len(eval_users) > 5000:
    eval_users = RNG.choice(eval_users, 5000, replace=False)
discount = 1 / np.log2(np.arange(2, K + 2))

def evaluate(score_fn, name, k=K, batch=250):
    P, Rc, N, H, recommended = [], [], [], [], set()
    for s in range(0, len(eval_users), batch):
        us = eval_users[s:s + batch]
        scores = score_fn(R_train[us]).astype(np.float64)
        seen = R_train[us].nonzero()
        scores[seen] = -np.inf                     # never recommend what they already liked
        top = np.argpartition(-scores, k, axis=1)[:, :k]
        order = np.take_along_axis(scores, top, 1).argsort(axis=1)[:, ::-1]
        top = np.take_along_axis(top, order, 1)
        for u, rec in zip(us, top):
            truth = test_items[u]
            hits = np.array([r in truth for r in rec], dtype=float)
            P.append(hits.sum() / k); Rc.append(hits.sum() / len(truth)); H.append(hits.any())
            idcg = discount[:min(len(truth), k)].sum()
            N.append((hits * discount).sum() / idcg)
            recommended.update(rec.tolist())
    return {"model": name, f"Precision@{k}": np.mean(P), f"Recall@{k}": np.mean(Rc),
            f"NDCG@{k}": np.mean(N), f"HitRate@{k}": np.mean(H), "Coverage": len(recommended) / len(item_ids)}

pop_fn = lambda Ru: np.tile(popularity, (Ru.shape[0], 1))
rand_fn = lambda Ru: RNG.random((Ru.shape[0], len(item_ids)))

In [ ]:
results = [evaluate(rand_fn, "Random"),
           evaluate(pop_fn, "Popularity"),
           evaluate(cb_scores, "Content-based"),
           evaluate(cf_scores, "Item-kNN CF")]
pd.DataFrame(results).set_index("model").round(4)

### Tuning α (the hybrid weight)

In [ ]:
alphas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sweep = pd.DataFrame([evaluate(lambda Ru, a=a: hybrid_scores(Ru, a), f"a={a}") for a in alphas])
sweep["alpha"] = alphas
BEST_ALPHA = float(sweep.loc[sweep[f"NDCG@{K}"].idxmax(), "alpha"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sweep.alpha, sweep[f"NDCG@{K}"], "o-", label=f"NDCG@{K}")
ax1.plot(sweep.alpha, sweep[f"Recall@{K}"], "s-", label=f"Recall@{K}")
ax1.set_xlabel("α  (0 = pure content, 1 = pure CF)"); ax1.set_ylabel("score"); ax1.legend(loc="upper left")
ax2 = ax1.twinx(); ax2.plot(sweep.alpha, sweep.Coverage, "^--", color="gray", label="Coverage"); ax2.set_ylabel("coverage")
ax1.axvline(BEST_ALPHA, color="red", ls=":"); plt.title(f"Best α = {BEST_ALPHA}"); plt.tight_layout(); plt.show()
sweep.round(4)

In [ ]:
results.append(evaluate(lambda Ru: hybrid_scores(Ru, BEST_ALPHA), f"Hybrid (α={BEST_ALPHA})"))
final = pd.DataFrame(results).set_index("model").round(4)
ax = final[[f"Precision@{K}", f"Recall@{K}", f"NDCG@{K}", f"HitRate@{K}"]].plot.bar(figsize=(11, 4), rot=0)
ax.set_title("Model comparison"); plt.tight_layout(); plt.show()
final

**How to read the results:**
- Everything should beat **Random** by a wide margin — that's the sanity check.
- **Popularity** is a strong accuracy baseline on Steam (everyone likes the same hits) but has near-zero **coverage**: it recommends the same 10 games to everyone.
- **Content-based** captures taste (a strategy player gets strategy games) and can recommend games nobody has reviewed yet (cold start), but ignores quality/popularity.
- **CF** captures real co-liking behaviour and is usually the strongest single model.
- The **hybrid** should match or beat CF on NDCG while keeping coverage higher than popularity — the best of both.

## 10b. Closing the gaps found in existing game recommenders

Our review of existing systems found four recurring gaps (see the write-up for sources):

| Gap in existing systems | What we add | How we measure it |
|---|---|---|
| Positive-only learning: "not recommended" reviews and dislikes are ignored | **Dislike-aware scoring**: never recommend a disliked game, and push down games similar to it | **DislikedHit@10**: % of players whose top 10 contains a game they actually disliked (lower is better) |
| Accuracy-first ranking: top 10 lists are full of near-duplicates | **Variety re-ranking (MMR)**: each next pick balances relevance against similarity to games already picked | **Diversity@10**: 1 − average content similarity between the 10 games (higher = more varied) |
| Popularity bias: hits crowd out smaller games | **Hidden-gems control**: a penalty on popularity | **Novelty@10** (how unpopular the picks are) and **LongTail@10** (% of picks outside the top-20% most-liked games) |
| No user constraints (price, year, rating, platform) | Filters in the app | Shown in the demo |

Negative reviews are split per player in half: one half is **known** (the system may use it), the other half is **hidden** and used only to check whether the system still recommends games the player disliked.

In [ ]:
# negative reviews for the players and games we kept, minus any game the player also liked
neg = neg_reviews[neg_reviews.user_id.astype(str).isin(user_idx) & neg_reviews.app_id.isin(app_to_idx)].drop_duplicates()
neg = pd.DataFrame({"u": neg.user_id.astype(str).map(user_idx).to_numpy(), "i": neg.app_id.map(app_to_idx).to_numpy()})
neg = neg.merge(inter[["u", "i"]], on=["u", "i"], how="left", indicator=True)
neg = neg[neg["_merge"] == "left_only"][["u", "i"]]
del neg_reviews

neg = neg.sample(frac=1.0, random_state=7)
neg["r"] = neg.groupby("u").cumcount(); neg["n"] = neg.groupby("u").u.transform("size")
known_neg = neg[neg.r < np.ceil(neg.n / 2)]
held_neg = neg[neg.r >= np.ceil(neg.n / 2)]

def to_binary(df):
    return sp.csr_matrix((np.ones(len(df), np.float32), (df.u.to_numpy(), df.i.to_numpy())),
                         shape=(len(user_ids), len(item_ids)))

D_known = to_binary(known_neg)          # dislikes the system may use
D_full = to_binary(neg)                 # all dislikes (for the exported model)
held_neg_items = held_neg.groupby("u").i.apply(set).to_dict()
n_eval_with_dislikes = sum(u in held_neg_items for u in eval_users)
print(f"Dislikes: {len(neg):,} from {neg.u.nunique():,} players | "
      f"{n_eval_with_dislikes:,} of the {len(eval_users):,} test players have hidden dislikes to check against")

In [ ]:
pop_norm = np.log1p(popularity) / np.log1p(popularity.max())   # 0 = niche ... 1 = most popular game
head = popularity >= np.quantile(popularity, 0.8)               # the 20% most-liked games
n_players = len(user_ids)

def dislike_penalty(D_users, alpha):
    # how close each game is to the games the player disliked (0..1), by players AND by content
    return alpha * minmax_rows(cf_scores(D_users)) + (1 - alpha) * minmax_rows(cb_scores(D_users))

def gap_scores(Ru, Du, alpha, use_dislikes=False, lam=0.5, gamma=0.0):
    s = hybrid_scores(Ru, alpha)
    if use_dislikes:
        s = s - lam * dislike_penalty(Du, alpha)
    if gamma:
        s = s - gamma * pop_norm[None, :]
    return s

def mmr(score_row, k, variety, n_cand=100):
    # Maximal Marginal Relevance: pick the next game by relevance minus similarity to games already picked
    cand = np.argpartition(-score_row, n_cand)[:n_cand]
    cand = cand[np.isfinite(score_row[cand])]
    rel = score_row[cand]; rel = (rel - rel.min()) / (rel.max() - rel.min() + 1e-9)
    sim = (X_content[cand] @ X_content[cand].T).toarray()
    chosen = [int(np.argmax(rel))]; closest = sim[chosen[0]].copy()
    while len(chosen) < min(k, len(cand)):
        val = (1 - variety) * rel - variety * closest
        val[chosen] = -np.inf
        j = int(np.argmax(val)); chosen.append(j); closest = np.maximum(closest, sim[j])
    return cand[chosen]

def evaluate_plus(name, use_dislikes=False, lam=0.5, variety=0.0, gamma=0.0, k=K, batch=250):
    P, Rc, N, H, DIV, NOV, TAIL, BAD, recommended = [], [], [], [], [], [], [], [], set()
    for s in range(0, len(eval_users), batch):
        us = eval_users[s:s + batch]
        scores = gap_scores(R_train[us], D_known[us], BEST_ALPHA, use_dislikes, lam, gamma).astype(np.float64)
        scores[R_train[us].nonzero()] = -np.inf
        if use_dislikes:
            scores[D_known[us].nonzero()] = -np.inf          # never recommend a game the player said they disliked
        for row, u in enumerate(us):
            sc = scores[row]
            if variety > 0:
                rec = mmr(sc, k, variety)
            else:
                rec = np.argpartition(-sc, k)[:k]; rec = rec[np.argsort(-sc[rec])]
            truth = test_items[u]
            hits = np.array([r in truth for r in rec], dtype=float)
            P.append(hits.sum() / k); Rc.append(hits.sum() / len(truth)); H.append(hits.any())
            N.append((hits * discount[:len(rec)]).sum() / discount[:min(len(truth), k)].sum())
            sim = (X_content[rec] @ X_content[rec].T).toarray()
            DIV.append(1 - (sim.sum() - np.trace(sim)) / (len(rec) * (len(rec) - 1)))
            NOV.append(np.mean(-np.log2(popularity[rec] / n_players)))
            TAIL.append(np.mean(~head[rec]))
            if u in held_neg_items:
                BAD.append(bool(set(rec.tolist()) & held_neg_items[u]))
            recommended.update(rec.tolist())
    return {"model": name, f"NDCG@{k}": np.mean(N), f"HitRate@{k}": np.mean(H), f"Recall@{k}": np.mean(Rc),
            "Coverage": len(recommended) / len(item_ids), f"Diversity@{k}": np.mean(DIV),
            f"Novelty@{k}": np.mean(NOV), f"LongTail@{k}": np.mean(TAIL),
            f"DislikedHit@{k}": np.mean(BAD) if BAD else np.nan}

In [ ]:
LAM, VARIETY, GAMMA = 0.5, 0.3, 0.2          # moderate default strengths; the app lets users change them
t0 = time.time()
gap_rows = [
    evaluate_plus("Hybrid (before)"),
    evaluate_plus("+ dislike-aware", use_dislikes=True, lam=LAM),
    evaluate_plus("+ variety (MMR)", variety=VARIETY),
    evaluate_plus("+ hidden gems", gamma=GAMMA),
    evaluate_plus("All three", use_dislikes=True, lam=LAM, variety=VARIETY, gamma=GAMMA),
]
gaps = pd.DataFrame(gap_rows).set_index("model")
print(f"Evaluated in {time.time() - t0:.0f}s")
gaps.round(4)

In [ ]:
# what each feature changed, relative to the hybrid before
change = ((gaps / gaps.loc["Hybrid (before)"]) - 1) * 100
fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for a, col, good in zip(ax, [f"NDCG@{K}", f"DislikedHit@{K}", f"Diversity@{K}", f"LongTail@{K}"],
                        ["higher", "lower", "higher", "higher"]):
    a.barh(gaps.index[::-1], gaps[col][::-1], color=["#c8321e" if i == 0 else "#8a8270" for i in range(len(gaps))][::-1])
    a.set_title(f"{col} ({good} is better)", fontsize=10)
plt.tight_layout(); plt.show()
change.round(1).rename(columns=lambda c: c + " Δ%")

**How to read this:**
- **+ dislike-aware** should lower **DislikedHit@10**: fewer players are shown a game they actually disliked. Accuracy should stay about the same.
- **+ variety (MMR)** should raise **Diversity@10**: fewer near-duplicate games in one list, usually at a small cost in NDCG.
- **+ hidden gems** should raise **Novelty@10** and **LongTail@10**: more smaller games, again at a small accuracy cost.
- This is the classic **accuracy-vs-diversity trade-off**. Instead of hard-coding one balance, the app exposes these as sliders so each player chooses.

## 10c. The "new game" test (cold start)

**Question:** when a game has just been released and nobody has reviewed it yet, can the system still recommend it to the right players?

**How we test it:**
1. Pick **500 games** at random and pretend they are brand new: **every review of them is removed from training**.
2. Rebuild the collaborative-filtering model without them. (The content model keeps their tags and descriptions — a new game has those on day one.)
3. For players who actually liked at least one of these games, check two things:
   - **New-releases shelf:** rank only the 500 new games. Are the ones the player liked near the top? (NDCG@10, HitRate@10)
   - **Normal list:** in the player's usual top 10 out of all games, how many are new games? (NewInTop10)

**Expected:** CF and Popularity can't tell new games apart (no reviews = no signal), so they score like Random. Content-based and the hybrid can, because they read the tags.

In [ ]:
COLD_N = 500
rng_cs = np.random.default_rng(11)
has_content = np.diff(X_content.indptr) > 0
eligible = np.flatnonzero(has_content & (popularity >= 20))
cold = np.sort(rng_cs.choice(eligible, min(COLD_N, len(eligible)), replace=False))
is_cold = np.zeros(len(item_ids), dtype=bool); is_cold[cold] = True

# training data with every review of the "new" games removed, and CF rebuilt without them
R_cs = to_matrix(train[~is_cold[train.i.to_numpy()]])
t0 = time.time()
S_cs = item_knn(R_cs, k=50)
pop_cs = np.asarray((R_cs > 0).sum(axis=0)).ravel().astype(float)   # popularity as the system would see it: 0 for new games
print(f"{len(cold)} games treated as new | CF rebuilt in {time.time() - t0:.0f}s")

# the answers: every new game each player actually liked
cold_truth = inter[is_cold[inter.i.to_numpy()]].groupby("u").i.apply(set).to_dict()
profile_size = np.diff(R_cs.indptr)
cs_users = np.array([u for u in cold_truth if profile_size[u] >= 3])
if len(cs_users) > 5000:
    cs_users = rng_cs.choice(cs_users, 5000, replace=False)
print(f"{len(cs_users):,} test players liked at least one of the new games")

In [ ]:
def cold_eval(name, score_fn, k=K, batch=250):
    N, H, Rc, NEW = [], [], [], []
    for s in range(0, len(cs_users), batch):
        us = cs_users[s:s + batch]
        Ru = R_cs[us]
        full = score_fn(Ru).astype(np.float64)
        full += rng_cs.random(full.shape) * 1e-9          # break exact ties fairly (e.g. all-zero scores)
        shelf_top = np.argsort(-full[:, cold], axis=1)[:, :k]   # (a) rank only the 500 new games
        full[Ru.nonzero()] = -np.inf
        all_top = np.argpartition(-full, k, axis=1)[:, :k]      # (b) the normal top 10 out of all games
        for row, u in enumerate(us):
            rec, truth = cold[shelf_top[row]], cold_truth[u]
            hits = np.array([r in truth for r in rec], dtype=float)
            H.append(hits.any()); Rc.append(hits.sum() / len(truth))
            N.append((hits * discount).sum() / discount[:min(len(truth), k)].sum())
            NEW.append(is_cold[all_top[row]].sum())
    return {"model": name, f"NDCG@{k} (new games)": np.mean(N), f"HitRate@{k} (new games)": np.mean(H),
            f"Recall@{k} (new games)": np.mean(Rc), "NewInTop10 (avg)": np.mean(NEW)}

cf_cs = lambda Ru: (Ru @ S_cs).toarray()
hyb_cs = lambda Ru: BEST_ALPHA * minmax_rows(cf_cs(Ru)) + (1 - BEST_ALPHA) * minmax_rows(cb_scores(Ru))

# fix: a new game has no CF score, so we ESTIMATE one from the 10 known games most similar to it in content
# ("players like you liked games like this one"), then blend as usual
known = np.flatnonzero(pop_cs > 0)
sim_new = (X_content[cold] @ X_content[known].T).toarray()             # new games x known games
nb = np.argpartition(-sim_new, 10, axis=1)[:, :10]                      # 10 closest known games per new game
w = np.take_along_axis(sim_new, nb, 1)
M = sp.csr_matrix((w.ravel(), (known[nb.ravel()], np.repeat(cold, 10))), shape=(len(item_ids), len(item_ids)))
M = M.multiply(1 / np.maximum(np.asarray(M.sum(axis=0)).ravel(), 1e-9)).tocsr()   # weighted average

def hyb_impute(Ru):
    cf = cf_cs(Ru)
    cf = cf + np.asarray(M.T @ cf.T).T          # new games get the weighted-average CF score of their neighbours
    return BEST_ALPHA * minmax_rows(cf) + (1 - BEST_ALPHA) * minmax_rows(cb_scores(Ru))

t0 = time.time()
cold_rows = [cold_eval("Random", lambda Ru: RNG.random((Ru.shape[0], len(item_ids)))),
             cold_eval("Popularity", lambda Ru: np.tile(pop_cs, (Ru.shape[0], 1))),
             cold_eval("Item-kNN CF", cf_cs),
             cold_eval("Content-based", cb_scores),
             cold_eval(f"Hybrid (α={BEST_ALPHA})", hyb_cs),
             cold_eval("Hybrid + CF estimate for new games", hyb_impute)]
cold_res = pd.DataFrame(cold_rows).set_index("model")
print(f"Evaluated in {time.time() - t0:.0f}s")

col = f"NDCG@{K} (new games)"
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
colors = ["#8a8270", "#8a8270", "#8a8270", "#1f4bd8", "#c8321e", "#c8321e"]
cold_res[col].plot.barh(ax=ax[0], color=colors); ax[0].invert_yaxis()
ax[0].set_title("New-releases shelf: ranking the 500 new games"); ax[0].set_xlabel(col)
cold_res["NewInTop10 (avg)"].plot.barh(ax=ax[1], color=colors); ax[1].invert_yaxis()
ax[1].set_title("Normal list: new games in each player's top 10"); ax[1].set_xlabel("new games per top 10 (avg)")
plt.tight_layout(); plt.show()
cold_res.round(4)

**How to read this:**
- **Random, Popularity and CF score about the same.** With no reviews, CF has no similar-player signal and Popularity sees every new game as having 0 fans, so they are effectively guessing.
- **Content-based and the hybrid score far higher.** They recognise a new game from its tags and description on day one.
- **On the new-releases shelf**, the plain hybrid scores the same as content-based: the CF part has nothing to say about new games, so the ranking comes from content. CF wins on known games, content rescues new ones — exactly why we built a hybrid.
- **Improvement — estimate CF for new games:** for each new game we find the 10 known games most similar to it in content and give it the weighted average of *their* CF scores ("players like you liked games like this one"). The hybrid can then use both parts for new games too, and it **ranks new games better than content alone**. Known games are untouched, so normal accuracy does not change.
- **NewInTop10** shows the other side: in the normal list, new games almost never reach the top 10 for CF or the hybrid, because a game with no (or only estimated) player data can't outscore proven hits. This is why real stores show new games on a separate **"New releases" shelf** — which is exactly the setting where our hybrid works and CF fails.

## 11. Demo — recommendations for a real user

In [ ]:
ALPHA = BEST_ALPHA
R_all = R_full.tocsr()

def recommend_for_user(user_id, k=10, alpha=ALPHA):
    Ru = R_all[user_idx[str(user_id)]]
    cf, cb = minmax_rows(cf_scores(Ru)), minmax_rows(cb_scores(Ru))
    s = (alpha * cf + (1 - alpha) * cb).ravel()
    s[Ru.indices] = -np.inf
    top = np.argsort(-s)[:k]
    return pd.DataFrame({"game": catalog.title.values[top], "score": s[top].round(3),
                         "cf": cf.ravel()[top].round(3), "content": cb.ravel()[top].round(3),
                         "tags": [", ".join(t[:4]) for t in catalog.tags.values[top]]})

demo_user = user_ids[int(RNG.integers(len(user_ids)))]
hist = inter[inter.u == user_idx[demo_user]].nlargest(8, "hours")
print(f"User {demo_user} — most-played liked games:")
for _, r in hist.iterrows():
    print(f"  {title_of[r.i]:<45} {r.hours:7.1f} h")
recommend_for_user(demo_user)

### Cold-start user: "I like these games"
A new user has no history — we build a pseudo-profile from a few games they pick (each treated as a positive review with ~10 hours played).

In [ ]:
def recommend_from_games(titles, k=10, alpha=ALPHA):
    idx = [find_game(t) for t in titles]
    Ru = sp.csr_matrix((np.full(len(idx), 1 + np.log1p(10), dtype=np.float32), ([0] * len(idx), idx)),
                       shape=(1, len(item_ids)))
    cf, cb = minmax_rows(cf_scores(Ru)), minmax_rows(cb_scores(Ru))
    s = (alpha * cf + (1 - alpha) * cb).ravel(); s[idx] = -np.inf
    top = np.argsort(-s)[:k]
    return pd.DataFrame({"game": catalog.title.values[top], "score": s[top].round(3),
                         "tags": [", ".join(t[:4]) for t in catalog.tags.values[top]]})

picks = catalog.nlargest(40, "players").title.sample(3, random_state=1).tolist()
print("Picked:", picks)
recommend_from_games(picks)

## 12. Export the model for the Streamlit app
We save only what the app needs: the catalogue, the content matrix, the CF neighbour matrix (re-trained on **all** data), and the histories of a sample of 20,000 users for the "existing user" demo (keeps the file small enough for GitHub).

In [ ]:
S_cf_full = item_knn(R_full, k=50)
demo_rows = np.sort(RNG.choice(len(user_ids), min(20000, len(user_ids)), replace=False))
model = {
    "catalog": catalog.assign(blurb=catalog.description.str.slice(0, 280))[     # blurb = short description for the app
                       ["item_id", "title", "tags", "genres", "developer", "date_release", "rating", "price_final",
                        "players", "median_hours", "blurb"] + [c for c in ["win", "mac", "linux", "steam_deck"] if c in catalog]
                       ].reset_index(drop=True),
    "X_content": X_content.tocsr(),
    "S_cf": S_cf_full,
    "R": R_full[demo_rows].tocsr(),
    "user_ids": [user_ids[i] for i in demo_rows],
    "weight_kind": "1+log1p(hours)",
    "best_alpha": BEST_ALPHA,
    "metrics": final.reset_index().to_dict("records"),
    "D": D_full[demo_rows].tocsr(),                                  # demo players' dislikes
    "gap_metrics": gaps.reset_index().to_dict("records"),
    "gap_defaults": {"lam": LAM, "variety": VARIETY, "gamma": GAMMA},
    "cold_metrics": cold_res.reset_index().to_dict("records"),
    "tfidf_tags": tfidf_tags, "tfidf_desc": tfidf_desc, "desc_weight": 0.5,   # for "describe the game you want" search
}
for v in (tfidf_tags, tfidf_desc):
    v.stop_words_ = None        # drop the list of discarded words: not needed to transform text, and large
joblib.dump(model, "steam_recommender_model.joblib", compress=3)
print(f"Saved steam_recommender_model.joblib ({os.path.getsize('steam_recommender_model.joblib')/1e6:.1f} MB)")
print("Next: put it next to app.py and run  streamlit run app.py")

In [ ]:
# In Colab: download the model file to your computer
try:
    from google.colab import files
    files.download("steam_recommender_model.joblib")
except ImportError:
    pass

## 13. Conclusion & future work
- A **hybrid** of item-kNN collaborative filtering on liked games (weighted by hours) and TF-IDF content similarity (tags + description) gives accurate *and* more diverse recommendations than either part alone or a popularity baseline.
- The **content** half handles **cold-start** games and new users; the **CF** half captures what players actually enjoy.

**Possible extensions:** ALS / matrix factorisation (`implicit` library); sentence-embeddings of descriptions; using negative reviews as explicit dislikes; diversity re-ranking (MMR); a time-based split using review dates.

**Dataset:** Anton Kozyriev, *Game Recommendations on Steam*, Kaggle.